# Supplementary Figure 2

In [ ]:
library(tidyverse)
library(data.table)
library(ggVennDiagram)
library(ggrepel)
library(patchwork)

out_dir <- "."

# Canonical exclusion list: sole-modification Trp C-mannosylation glycopeptiforms mislabeled
w_sole_exclude_gpids <- readLines("../../2_main_figures/Figure1/data/w_sole_exclude_gpids.txt")
stopifnot(length(w_sole_exclude_gpids) == 54L)
qc_remove_gpids <- readLines("../../2_main_figures/Figure1/data/qc_remove_gpids.txt")
coding_drop_gpids <- readLines("../../2_main_figures/Figure1/data/coding_variant_drop_gpids.txt")
stopifnot(length(coding_drop_gpids) == 21L)
excluded_gpids <- c(qc_remove_gpids, w_sole_exclude_gpids, coding_drop_gpids)

suppressMessages(library(colocboost))
tmem_id <- "ENSG00000106460"
tmem_cos <- function(rds) {
  cb <- readRDS(rds)[[1]]
  cb$cos_details$cos_weights <- lapply(cb$cos_details$cos_weights, function(w) { colnames(w) <- sub("^Y", "outcome", colnames(w)); w })
  cs <- suppressMessages(suppressWarnings(get_robust_colocalization(cb, cos_npc_cutoff = 0.2, npc_outcome_cutoff = 0.2, pvalue_cutoff = 1e-4)))$cos_summary
  cs$gene <- tmem_id
  as.data.table(cs[grepl("gp_", cs$colocalized_outcomes), ])
}
tmem_dir <- "data_output/TMEM106B_rerun_202609/glycoQTL_tmem106b"



In [ ]:
# Panel a: TensorQTL versus SuSiE overlap
tensorqtl_file <- "data_output/TMEM106B_rerun_202609/tensorqtl_unadjusted/unadj_glyco.sorted.bed.cis_qtl_regional_significance.v202609.tsv.gz"
susie_merged_file <- "data_output/finemapping/ROSMAP_gpQTL.exported.toploci.bed.gz"
stopifnot(file.exists(tensorqtl_file), file.exists(path.expand(susie_merged_file)))

tqtl <- read.table(
  gzfile(tensorqtl_file),
  header = TRUE,
  sep = "\t",
  stringsAsFactors = FALSE
)
susie_all <- fread(susie_merged_file) %>%
  filter(str_detect(event_ID, "unadjust")) %>%
  mutate(molecular_trait_object_id = str_extract(event_ID, "gp_[0-9]+")) %>%
  filter(MAF >= 0.05, !molecular_trait_object_id %in% excluded_gpids)

susie_summary <- susie_all %>%
  group_by(molecular_trait_object_id) %>%
  summarise(has_cs95 = any(cs_coverage_0.95 > 0, na.rm = TRUE), .groups = "drop")

setA <- tqtl %>%
  filter(fdr_perm < 0.05, !molecular_trait_object_id %in% excluded_gpids) %>%
  pull(molecular_trait_object_id) %>%
  unique()
setC <- susie_summary %>%
  filter(has_cs95) %>%
  pull(molecular_trait_object_id) %>%
  unique()

stopifnot(
  length(setdiff(setA, setC)) == 57L,
  length(intersect(setA, setC)) == 130L,
  length(setdiff(setC, setA)) == 76L
)

venn_list <- list(
  "TensorQTL\n(perm q < 0.05)" = setA,
  "SuSiE\n(95% CS)" = setC
)
f2a <- ggVennDiagram(
  venn_list,
  label = "count",
  label_alpha = 0,
  category.names = c("TensorQTL ", "SuSiE ")
) +
  coord_flip() +
  scale_fill_gradient(low = "#F7FBFF", high = "#2166AC") +
  scale_color_manual(values = c("#D6604D", "#2166AC")) +
  scale_y_continuous(expand = expansion(mult = c(0.05, 0.20))) +
  theme(
    legend.position = "none",
    plot.title = element_text(face = "bold", hjust = 0.5)
  )


In [ ]:
# Panel b: validate and plot the requested glycoQTL counts
pqtl_coloc_file <- "data_output/colocalization/maf05/glycoQTL_maf05/pQTL_glycoQTL_unadjusted_summary_pv1e4.tsv"
stopifnot(file.exists(path.expand(pqtl_coloc_file)))

pqtl_coloc <- fread(pqtl_coloc_file)
stopifnot(all(pqtl_coloc$cos_npc >= 0.2 & pqtl_coloc$min_npc_outcome > 0.2))
stopifnot("colocalized_outcomes" %in% names(pqtl_coloc))

tm_b <- tmem_cos(file.path(tmem_dir, "glycoQTL_unadjusted_AD/colocboost/unadj_glyco_AD.chr7_ENSG00000106460.cb_xqtl.rds"))
cat("Panel b, TMEM106B rows: old", sum(pqtl_coloc$gene == tmem_id), "-> new", nrow(tm_b), ":", tm_b$colocalized_outcomes, "\n")
pqtl_coloc <- rbind(pqtl_coloc[gene != tmem_id], tm_b[, intersect(names(pqtl_coloc), names(tm_b)), with = FALSE], fill = TRUE)

outcome_tokens <- str_split(pqtl_coloc$colocalized_outcomes, ";\\s*")
excl_gp_token <- function(tok) {
  gp <- str_remove(tok, "DLPFC_Klein_gpQTL_unadjusted_")
  str_detect(tok, "^DLPFC_Klein_gpQTL_unadjusted_gp_[0-9]+$") & gp %in% excluded_gpids
}
trimmed_tokens <- lapply(outcome_tokens, function(tok) tok[!excl_gp_token(tok)])
n_trimmed <- sum(lengths(outcome_tokens)) - sum(lengths(trimmed_tokens))
keep_row <- lengths(trimmed_tokens) >= 2
cat(sprintf(
  "Panel b: trimmed %d excluded-gpID token(s); %d of %d rows dropped entirely (<2 members remain after trimming)\n",
  n_trimmed, sum(!keep_row), nrow(pqtl_coloc)
))
pqtl_coloc <- pqtl_coloc[keep_row, ]
pqtl_coloc[, colocalized_outcomes := vapply(trimmed_tokens[keep_row], paste, character(1), collapse = "; ")]

has_glyco <- str_detect(
  pqtl_coloc$colocalized_outcomes,
  fixed("DLPFC_Klein_gpQTL_unadjusted")
)
has_pqtl <- str_detect(
  pqtl_coloc$colocalized_outcomes,
  fixed("DLPFC_Bennett_pQTL")
)

n_with_pqtl <- sum(has_glyco & has_pqtl, na.rm = TRUE)
n_all_glyco <- sum(has_glyco, na.rm = TRUE)
n_without_pqtl <- n_all_glyco - n_with_pqtl
stopifnot(identical(c(n_with_pqtl, n_without_pqtl), c(108L, 143L)))   # previous version: 108 / 148

bar_df <- data.frame(
  status = factor(
    c("Colocalized with pQTL", "Not colocalized with pQTL"),
    levels = c("Colocalized with pQTL", "Not colocalized with pQTL")
  ),
  count = c(n_with_pqtl, n_without_pqtl)
)

f2b <- ggplot(bar_df, aes(x = status, y = count, fill = status)) +
  geom_col(width = 0.62, color = "black", linewidth = 0.3) +
  geom_text(aes(label = count), vjust = -0.45, size = 4.2) +
  scale_fill_manual(
    values = c(
      "Colocalized with pQTL" = "#4C72B0",
      "Not colocalized with pQTL" = "grey70"
    ),
    guide = "none"
  ) +
  scale_y_continuous(
    breaks = seq(0, 160, 40),
    limits = c(0, 170),
    expand = expansion(mult = c(0, 0))
  ) +
  scale_x_discrete(labels = c(
    "Colocalized with pQTL" = "Colocalized\nwith pQTL",
    "Not colocalized with pQTL" = "Not colocalized\nwith pQTL"
  )) +
  labs(x = NULL, y = "Number of glycoQTLs") +
  theme_classic(base_size = 12) +
  theme(
    panel.grid = element_blank(),
    axis.line = element_line(color = "black", linewidth = 0.35),
    axis.ticks = element_line(color = "black", linewidth = 0.35),
    axis.text = element_text(color = "black"),
    axis.text.x = element_text(size = 9.5, margin = margin(t = 5)),
    axis.title.y = element_text(size = 11, margin = margin(r = 4)),
    plot.margin = margin(5, 8, 5, 8)
  )


In [ ]:
# Panel c: colocalized glycoQTL/eQTL signals versus detected eGenes
fm_file <- "FunGen_xQTL_combined_resources/single_context_finemapping_combined.bed.gz"
eqtl_coloc_file <- "data_output/colocalization/maf05/glycoQTL_maf05/glycoQTL_eQTL_summary_pv1e4.tsv"
stopifnot(file.exists(fm_file), file.exists(eqtl_coloc_file))

cells_per_donor <- c(Exc = 1618, Oli = 736, Inh = 561, Ast = 497, Mic = 192, OPC = 124)
fm <- fread(
  fm_file,
  select = c("gene_ID", "cs_coverage_0.95", "Cohort_Data_Type_Modality")
)
egene <- fm[
  str_detect(
    Cohort_Data_Type_Modality,
    "^ROSMAP_eQTL_(Ast|Exc|Inh|Mic|OPC|Oli)_DeJager$"
  ) & `cs_coverage_0.95` > 0,
  .(n_eGene = uniqueN(gene_ID)),
  by = .(
    cell = str_match(
      Cohort_Data_Type_Modality,
      "ROSMAP_eQTL_([A-Za-z]+)_DeJager"
    )[, 2]
  )
]

gp_e <- fread(eqtl_coloc_file)
stopifnot(all(gp_e$cos_npc >= 0.2 & gp_e$min_npc_outcome > 0.2))
tm_c <- tmem_cos(file.path(tmem_dir, "glycoQTL_unadjusted_eQTL/colocboost/eQTL_gpQTL.chr7_ENSG00000106460.cb_xqtl.rds"))
cat("Panel c, TMEM106B rows: old", sum(gp_e$gene == tmem_id), "-> new", nrow(tm_c), ":", tm_c$colocalized_outcomes, "\n")
gp_e <- rbind(gp_e[gene != tmem_id], tm_c[, intersect(names(gp_e), names(tm_c)), with = FALSE], fill = TRUE)

outcome_tokens_e <- str_split(gp_e$colocalized_outcomes, ";\\s*")
trimmed_tokens_e <- lapply(outcome_tokens_e, function(tok) tok[!excl_gp_token(tok)])
n_trimmed_e <- sum(lengths(outcome_tokens_e)) - sum(lengths(trimmed_tokens_e))
keep_row_e <- lengths(trimmed_tokens_e) >= 2
cat(sprintf(
  "Panel c: trimmed %d excluded-gpID token(s); %d of %d rows dropped entirely (<2 members remain after trimming)\n",
  n_trimmed_e, sum(!keep_row_e), nrow(gp_e)
))
gp_e <- gp_e[keep_row_e]
gp_e[, colocalized_outcomes := vapply(trimmed_tokens_e[keep_row_e], paste, character(1), collapse = "; ")]

cos_ct <- rbindlist(lapply(names(cells_per_donor), function(cell_type) {
  data.table(
    cell = cell_type,
    n_CoS = sum(str_detect(gp_e$colocalized_outcomes, fixed(cell_type)), na.rm = TRUE)
  )
}))

sup <- Reduce(
  function(x, y) merge(x, y, by = "cell"),
  list(
    data.table(cell = names(cells_per_donor), cells = as.integer(cells_per_donor)),
    egene,
    cos_ct
  )
)
sup <- sup[order(-cells)]
stopifnot(nrow(sup) == 6L, !anyNA(sup[, .(n_eGene, n_CoS)]))
print(sup)

ct_cols <- c(
  Exc = "#4C72B0",
  Inh = "#55A868",
  Oli = "#8172B3",
  Ast = "#C44E52",
  Mic = "#CCB974",
  OPC = "#64B5CD"
)
base_thm <- theme_classic(base_size = 13) +
  theme(legend.position = "none", axis.text = element_text(color = "black"))
r_lab <- function(x, y) sprintf("r = %.2f", cor(x, y))

f2c <- ggplot(sup, aes(n_eGene, n_CoS)) +
  geom_smooth(method = "lm", se = TRUE, color = "grey60", fill = "grey85", linewidth = 0.6) +
  geom_point(aes(color = cell), size = 3.5) +
  geom_text_repel(aes(label = cell), size = 4, seed = 1) +
  scale_color_manual(values = ct_cols) +
  annotate(
    "text",
    x = Inf,
    y = -Inf,
    hjust = 1.1,
    vjust = -0.8,
    label = r_lab(sup$n_eGene, sup$n_CoS),
    size = 4.2
  ) +
  labs(
    x = "Detected eGenes",
    y = "Colocalized glycoQTLs and eQTLs"
  ) +
  base_thm


In [ ]:
fig2 <- (f2a | f2b | f2c) +
  plot_layout(widths = c(1.30, 0.72, 1.10)) +
  plot_annotation(tag_levels = "a") &
  theme(plot.tag = element_text(face = "bold", size = 16))

ggsave(
  file.path(out_dir, "Supplementary_Figure2.pdf"),
  fig2,
  width = 11.5,
  height = 4.0,
  device = cairo_pdf
)

cat(sprintf("Panel b counts: with pQTL = %d; without pQTL = %d\n", n_with_pqtl, n_without_pqtl))
cat(sprintf("Panel c cell types = %d; r = %.3f\n", nrow(sup), cor(sup$n_eGene, sup$n_CoS)))
